<a href="https://colab.research.google.com/github/maqsoodahmadkhan3982-dotcom/Materials-Properties-Response/blob/main/Phase_Boundary_%26_Mapping_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

class PhasePredictor:
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.le = LabelEncoder()
        self.feature_cols = []
        self.is_trained = False

    def parse_composition(self, value):
        """Helper to parse composition strings into dictionaries."""
        try:
            if isinstance(value, str):
                return ast.literal_eval(value)
            return value
        except:
            return {}

    def prepare_data(self, region_path, composition_path):
        """
        Combines data from both the phase region dataset and the composition dataset.
        """
        print(f"Loading datasets...")
        df_region = pd.read_csv(region_path)
        df_comp = pd.read_csv(composition_path)

        # 1. Process Phase Region Dataset (The 'Dense' map)
        region_features = df_region['Composition'].apply(self.parse_composition).apply(pd.Series)
        region_labels = df_region['Phase_type']

        # 2. Process Composition Dataset (The 'Stable Phase' list)
        # Note: This dataset uses 'Alphabetical formula' which we need to parse into percentages
        # For simplicity in this demo, we prioritize the explicitly structured region_features
        # but we combine the unique phase labels from both to ensure the encoder is complete.

        all_labels = pd.concat([region_labels, df_comp['QC, AC type'].dropna()])
        self.le.fit(all_labels.astype(str))

        # We fill NaNs with 0 to handle elements not present in specific systems
        X = region_features.fillna(0)
        y = self.le.transform(region_labels.astype(str))

        self.feature_cols = X.columns.tolist()
        return X.values, y

    def train(self, X, y):
        print(f"Training model on {len(X)} samples with features: {self.feature_cols}")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.model.fit(X_train, y_train)
        score = self.model.score(X_test, y_test)
        print(f"Model trained. Validation Accuracy: {score:.2%}")
        self.is_trained = True

    def predict_phase(self, input_features):
        """
        Takes a dictionary of features and returns the predicted phase and confidence.
        """
        if not self.is_trained:
            return {"Error": "Model not trained yet."}

        # Align input with the training feature columns
        feat_vector = [input_features.get(col, 0) for col in self.feature_cols]
        feat_vector = np.array(feat_vector).reshape(1, -1)

        prediction = self.model.predict(feat_vector)
        probabilities = self.model.predict_proba(feat_vector)

        phase = self.le.inverse_transform(prediction)[0]
        confidence = np.max(probabilities)

        # Boundary logic: Confidence < 0.6 suggests we are between phase regions
        is_boundary = "Yes (Potential Phase Boundary)" if confidence < 0.6 else "No (Stable Core Region)"

        return {
            "Predicted Phase": phase,
            "Confidence Score": f"{confidence:.2%}",
            "Boundary Proximity": is_boundary
        }

# --- Execution Block ---
if __name__ == "__main__":
    predictor = PhasePredictor()

    # File names as requested
    region_file = 'phase_region_dataset.csv'
    comp_file = 'composition_dataset.csv'

    try:
        # 1. Load and Prepare from both files
        X, y = predictor.prepare_data(region_file, comp_file)

        # 2. Train
        predictor.train(X, y)

        # 3. Predict for a specific composition (Example: Al-Au-Co system)
        # You can change these values to test different points in the phase diagram
        test_composition = {'Al': 75.0, 'Au': 12.5, 'Co': 12.5}

        print("\n--- Output for Input Composition ---")
        result = predictor.predict_phase(test_composition)
        for key, value in result.items():
            print(f"{key}: {value}")

    except FileNotFoundError as e:
        print(f"Error: Could not find file. {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")